# System Prompts (Course: `002_system_prompt.ipynb`)

System prompts allow us to customize Claude's tone, style, and persona of response.

### Case Study: Math Tutor Specialist
- **Goal**: Guide a student to solve math equations without directly giving away the answer.
- **Responses should**:
  - Initially only give hints
  - Patiently walk the student through step-by-step
  - Show similar problem examples
- **Responses should NOT**:
  - Immediately give a direct answer
  - Tell the student to use a calculator

In [ ]:
# Install dependencies
%pip install anthropic python-dotenv

In [ ]:
# Load environment variables
from dotenv import load_dotenv
load_dotenv()

In [ ]:
# Create API client
from anthropic import Anthropic

client = Anthropic()
model = "claude-3-5-sonnet-latest"

## 1. Initial Test (Without System Prompt)
Without a system prompt, Claude defaults to providing a full, direct mathematical solution immediately.

In [ ]:
# Helper functions (basic version)
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages):
    message = client.messages.create(
        model=model,
        max_tokens=1000,
        messages=messages,
    )
    return message.content[0].text

In [ ]:
messages = []
add_user_message(messages, "How do I solve 5x+3=2 for x?")
answer = chat(messages)
answer

## 2. Refactoring `chat()` for Dynamic System Prompts

> **Crucial Gotcha**: In the Anthropic API, passing `system=None` throws an error! 
> We must dynamically assemble a `params` dictionary and only include the `"system"` key if a system prompt was actually supplied.

In [ ]:
def chat(messages, system=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

## 3. Testing with the Math Tutor System Prompt
With the system prompt, Claude shifts into a tutoring persona, asking guiding questions instead of giving away the answer.

In [ ]:
messages = []

system = """
You are a patient math tutor.
Do not directly answer a student's questions.
Guide them to a solution step by step.
"""

add_user_message(messages, "How do I solve 5x+3=2 for x?")
answer = chat(messages, system=system)
answer